# Azurity Pharmaceuticals - Spark Connect Analytics

This notebook demonstrates **Snowpark Connect for Spark** (SCOS) - allowing teams with existing PySpark workloads to run Spark code directly against Snowflake data.

**Key Benefits**:
- Use familiar PySpark API without data movement
- SQL pushdown to Snowflake warehouse for optimized performance
- Leverage existing Spark expertise while modernizing on Snowflake

**Use Case**: Large-scale HCP (Healthcare Provider) analytics and aggregations

## 1. Setup Spark Connect in Container Runtime

Container Runtime doesn't include a JVM by default. We use `jdk4py` to provide a portable Java runtime.

In [ ]:
!pip install snowpark-connect[jdk] --quiet

In [ ]:
from snowflake import snowpark_connect
from pyspark.sql.functions import col, avg, count, sum as sum_

# Initialize Spark session via Snowpark Connect
spark = snowpark_connect.server.init_spark_session()
print(f"Spark Session: {spark}")
print(f"Spark Version: {spark.version}")

In [ ]:
from snowflake.snowpark.context import get_active_session
session = get_active_session()

session.sql("USE DATABASE AZURITY_DEMO_DB").collect()
session.sql("USE SCHEMA RAW").collect()
session.sql("USE WAREHOUSE AZURITY_ML_WH").collect()
print("Context set to AZURITY_DEMO_DB.RAW")

## 2. Access Snowflake Tables via Spark

Snowpark Connect allows direct table access using PySpark DataFrame API.

In [ ]:
prescriptions_df = spark.sql("SELECT * FROM AZURITY_DEMO_DB.RAW.PRESCRIPTIONS")
hcps_df = spark.sql("SELECT * FROM AZURITY_DEMO_DB.RAW.HEALTHCARE_PROVIDERS")
products_df = spark.sql("SELECT * FROM AZURITY_DEMO_DB.RAW.PRODUCTS")

print(f"Prescriptions: {prescriptions_df.count():,} rows")
print(f"Healthcare Providers: {hcps_df.count():,} rows")
print(f"Products: {products_df.count():,} rows")

In [ ]:
print("Prescriptions Schema:")
prescriptions_df.printSchema()

## 3. Large-Scale HCP Analytics with PySpark

Demonstrate familiar Spark operations that push down to Snowflake.

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [ ]:
hcp_volume = prescriptions_df.groupBy("HCP_ID", "PRODUCT_ID") \
    .agg(
        F.sum("QUANTITY").alias("TOTAL_VOLUME"),
        F.count("*").alias("RX_COUNT"),
        F.min("FILL_DATE").alias("FIRST_RX"),
        F.max("FILL_DATE").alias("LAST_RX")
    ) \
    .orderBy(F.desc("TOTAL_VOLUME"))

print("Top HCPs by Prescription Volume:")
hcp_volume.show(10)

In [ ]:
window_spec = Window.partitionBy("PRODUCT_ID").orderBy(F.desc("TOTAL_VOLUME"))

hcp_ranked = hcp_volume \
    .withColumn("RANK", F.dense_rank().over(window_spec)) \
    .withColumn("PCT_OF_PRODUCT", 
                F.round(F.col("TOTAL_VOLUME") * 100.0 / 
                       F.sum("TOTAL_VOLUME").over(Window.partitionBy("PRODUCT_ID")), 2))

print("Top 3 HCPs per Product:")
hcp_ranked.filter(F.col("RANK") <= 3).orderBy("PRODUCT_ID", "RANK").show(15)

## 4. Complex Joins and Enrichment

Join multiple large tables - Snowpark Connect pushes this to Snowflake's engine.

In [ ]:
enriched_rx = prescriptions_df \
    .join(hcps_df, "HCP_ID") \
    .join(products_df, "PRODUCT_ID") \
    .select(
        prescriptions_df.RX_ID,
        prescriptions_df.FILL_DATE,
        prescriptions_df.QUANTITY,
        products_df.PRODUCT_NAME,
        products_df.THERAPEUTIC_AREA,
        hcps_df.SPECIALTY,
        hcps_df.DECILE,
        hcps_df.STATE,
        hcps_df.REGION
    )

print(f"Enriched dataset: {enriched_rx.count():,} rows")
enriched_rx.show(5)

In [ ]:
regional_analysis = enriched_rx \
    .groupBy("REGION", "THERAPEUTIC_AREA") \
    .agg(
        F.sum("QUANTITY").alias("TOTAL_UNITS"),
        F.countDistinct("RX_ID").alias("UNIQUE_RX"),
        F.avg("DECILE").alias("AVG_HCP_DECILE")
    ) \
    .orderBy(F.desc("TOTAL_UNITS"))

print("Regional Performance by Therapeutic Area:")
regional_analysis.show(20)

## 5. Time Series Analysis

In [ ]:
monthly_trends = enriched_rx \
    .withColumn("MONTH", F.date_trunc("month", F.col("FILL_DATE"))) \
    .groupBy("MONTH", "PRODUCT_NAME") \
    .agg(
        F.sum("QUANTITY").alias("MONTHLY_VOLUME"),
        F.countDistinct("RX_ID").alias("UNIQUE_RX")
    ) \
    .orderBy("PRODUCT_NAME", "MONTH")

print("Monthly Volume Trends:")
monthly_trends.show(20)

## 6. Write Results Back to Snowflake

Save Spark DataFrame results as Snowflake tables.

In [ ]:
hcp_summary = hcp_volume.join(hcps_df, "HCP_ID") \
    .select(
        "HCP_ID",
        "PRODUCT_ID",
        "SPECIALTY",
        "DECILE",
        "STATE",
        "REGION",
        "TOTAL_VOLUME",
        "RX_COUNT",
        "FIRST_RX",
        "LAST_RX"
    )

hcp_summary_sf = session.create_dataframe(hcp_summary.toPandas())
hcp_summary_sf.write.mode("overwrite").save_as_table("AZURITY_DEMO_DB.ML.HCP_PRODUCT_SUMMARY")

print("Created table: AZURITY_DEMO_DB.ML.HCP_PRODUCT_SUMMARY")

In [ ]:
verify = session.table("AZURITY_DEMO_DB.ML.HCP_PRODUCT_SUMMARY")
print(f"Verified row count: {verify.count():,}")
verify.limit(5).show()

## 7. Understanding SQL Pushdown

Spark operations are translated to SQL and executed in Snowflake's engine.

In [ ]:
complex_query = prescriptions_df \
    .filter(F.col("QUANTITY") > 10) \
    .groupBy("REGION") \
    .agg(F.sum("QUANTITY").alias("TOTAL"))

print("Spark Execution Plan (demonstrates pushdown):")
complex_query.explain()

## 8. Summary

This notebook demonstrated:

| Capability | What We Showed |
|------------|----------------|
| **Spark Connect Setup** | jdk4py for JVM in Container Runtime |
| **Data Access** | spark.sql() to read Snowflake tables |
| **PySpark Operations** | groupBy, agg, join, window functions |
| **SQL Pushdown** | Operations translated to Snowflake SQL |
| **Write Back** | Save Spark results as Snowflake tables |

### Key Talking Points
1. **Migration Path**: Teams with existing PySpark code can run against Snowflake without rewriting
2. **No Data Movement**: Data stays in Snowflake; only query plans are exchanged
3. **Performance**: SQL pushdown leverages Snowflake's optimized engine

In [ ]:
print("=" * 60)
print("SPARK CONNECT DEMO COMPLETE")
print("=" * 60)
print(f"\nSpark Version: {spark.version}")
print(f"Tables Created: AZURITY_DEMO_DB.ML.HCP_PRODUCT_SUMMARY")
print(f"\nKey Pattern: Use familiar PySpark API while Snowflake handles compute.")